# Calculate 2025's CTR and CPC

*Data Analysis in Python · Topic 4.1*

The paid search team is writing its 2025 review and needs the year's spend, CTR and CPC for Google Ads, and the days with the most expensive clicks.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell and the cell that reads the file into `ads`.
2. In the first cell marked ✏️, create `ads_2025`: a copy of the rows dated in 2025. Add three columns to it:
   - `spend`: the cost as a number of dollars, such as `1.25` (`cost_micros` counts millionths of a dollar, so `1250000` is `1.25`)
   - `ctr`: clicks divided by impressions
   - `cpc`: spend divided by clicks
3. In the second cell marked ✏️, create:
   - `total_spend`: the year's spend as a number, rounded to 2 decimals (`198630.66`, not the text `"$198,630.66"`)
   - `overall_ctr`: the year's CTR, as a fraction (0.0121, not 1.21)
   - `overall_cpc`: the year's CPC as a number (`0.5755`, not the text `"$0.58"`)
4. In the last cell marked ✏️, create `top_cpc`: the five rows with the highest CPC, with the columns `date`, `campaign_name`, `clicks`, `spend` and `cpc`. Look at their clicks.
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `ads_2025` has 3,796 rows and the columns `spend`, `ctr` and `cpc`. `total_spend` is `198630.66`, `overall_ctr` is about `0.0121` (1.21%) and `overall_cpc` about `0.5755`, which reads as $0.58. `top_cpc` has five rows, and the first is February 8, 2025, `2025_ca_search_brand`, with a CPC of $1.32.

## Parameters

In [ ]:
import pandas as pd

FILE_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/google_ads_spend.csv"

In [ ]:
ads = pd.read_csv(FILE_URL, parse_dates=["date"])
ads.shape

## Metrics per row

In [ ]:
ads_2025 = ads[ads["date"].dt.strftime("%Y") == "2025"].copy()
ads_2025["spend"] = ads_2025["cost_micros"] / 1_000_000
ads_2025["ctr"] = ads_2025["clicks"] / ads_2025["impressions"]
ads_2025["cpc"] = ads_2025["spend"] / ads_2025["clicks"]
ads_2025.head()

## The year

In [ ]:
total_spend = round(ads_2025["spend"].sum(), 2)
overall_ctr = ads_2025["clicks"].sum() / ads_2025["impressions"].sum()
overall_cpc = ads_2025["spend"].sum() / ads_2025["clicks"].sum()
f"Spend ${total_spend:,.2f}, CTR {overall_ctr:.2%}, CPC ${overall_cpc:.2f}"

In [ ]:
top_cpc = ads_2025.nlargest(5, "cpc")[["date", "campaign_name", "clicks", "spend", "cpc"]]
top_cpc

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    ads_2025 = found.get("ads_2025")
    if not isinstance(ads_2025, pd.DataFrame):
        results.append((False, "ads_2025 isn't a DataFrame yet. Keep the rows dated in 2025: ads[ads[\"date\"].dt.strftime(\"%Y\") == \"2025\"].copy()."))
    elif len(ads_2025) != 3796:
        results.append((False, f"ads_2025 has {len(ads_2025):,} rows, but 2025 has 3,796. Keep the rows whose date is in 2025, and nothing else."))
    elif missing := [column for column in ["spend", "ctr", "cpc"] if column not in ads_2025.columns]:
        results.append((False, f"ads_2025 has no {', '.join(missing)} column yet. Add each one with ads_2025[\"spend\"] = ..."))
    elif abs(ads_2025["spend"].sum() - 198630660000) < 1:
        results.append((False, "ads_2025's spend is still in micros: the year's total is 198,630,660,000. Divide cost_micros by 1_000_000."))
    elif abs(ads_2025["spend"].sum() - 198630.66) > 0.005:
        results.append((False, f"ads_2025's spend adds up to {ads_2025['spend'].sum():,.2f}, but the year's spend is 198,630.66. spend is cost_micros / 1_000_000."))
    elif not ((ads_2025["ctr"] - ads_2025["clicks"] / ads_2025["impressions"]).abs() < 1e-9).all():
        results.append((False, "ads_2025's ctr isn't clicks / impressions on every row. Keep it a fraction: don't multiply by 100."))
    elif not ((ads_2025["cpc"] - ads_2025["spend"] / ads_2025["clicks"]).abs() < 1e-9).all():
        results.append((False, "ads_2025's cpc isn't spend / clicks on every row. Use the spend column, not cost_micros."))
    else:
        results.append((True, "ads_2025 has 2025's 3,796 rows, with spend converted from micros, and ctr and cpc on every row."))

    total_spend = found.get("total_spend")
    if isinstance(total_spend, (pd.DataFrame, pd.Series)):
        results.append((False, f"total_spend is a {type(total_spend).__name__}, not a number. Total one column: round(ads_2025[\"spend\"].sum(), 2)."))
    elif isinstance(total_spend, str):
        results.append((False, f"total_spend is the text {total_spend!r}. Keep it a number, 198630.66, and format it only when you show it, such as f\"${{total_spend:,.2f}}\"."))
    elif pd.api.types.is_number(total_spend) and abs(total_spend - 198630.66) < 0.005:
        results.append((True, "Google Ads spent $198,630.66 in 2025."))
    elif pd.api.types.is_number(total_spend) and abs(total_spend - 198630660000) < 1:
        results.append((False, "total_spend is 198,630,660,000: the micros, summed. Total the spend column instead."))
    else:
        results.append((False, f"total_spend should be round(ads_2025[\"spend\"].sum(), 2). It is {total_spend!r}."))

    ctr = found.get("overall_ctr")
    cpc = found.get("overall_cpc")
    if tables := [name for name, value in [("overall_ctr", ctr), ("overall_cpc", cpc)] if isinstance(value, (pd.DataFrame, pd.Series))]:
        results.append((False, f"{' and '.join(tables)} should be numbers, not tables. Divide two totals: ads_2025[\"clicks\"].sum() / ads_2025[\"impressions\"].sum()."))
    elif texts := [name for name, value in [("overall_ctr", ctr), ("overall_cpc", cpc)] if isinstance(value, str)]:
        results.append((False, f"{' and '.join(texts)} should be numbers, not text. Keep them numbers, such as 0.5755, and format them only when you show them: f\"{{overall_ctr:.2%}}\" and f\"${{overall_cpc:.2f}}\"."))
    elif not pd.api.types.is_number(ctr) or not pd.api.types.is_number(cpc):
        results.append((False, f"overall_ctr and overall_cpc should be numbers. They are {ctr!r} and {cpc!r}."))
    elif abs(ctr - 0.029468) < 0.00005:
        results.append((False, "overall_ctr is 2.95%, the average of the rows' CTRs. The year's CTR is the year's clicks over its impressions: ads_2025[\"clicks\"].sum() / ads_2025[\"impressions\"].sum(). Topic 4.4 shows why the two differ."))
    elif abs(ctr - 1.2069) < 0.005:
        results.append((False, "overall_ctr is 1.21: a percentage. Keep it a fraction, 0.0121, and format it when you show it: f\"{overall_ctr:.2%}\"."))
    elif abs(ctr - 0.012069) > 0.00005:
        results.append((False, f"overall_ctr should be the year's clicks over its impressions, about 0.0121. It is {ctr!r}."))
    elif abs(cpc - 0.4967) < 0.0005:
        results.append((False, "overall_cpc is $0.50, the average of the rows' CPCs. The year's CPC is its spend over its clicks: ads_2025[\"spend\"].sum() / ads_2025[\"clicks\"].sum()."))
    elif abs(cpc - 575452.48) < 1:
        results.append((False, "overall_cpc is $575,452: cost_micros over clicks. Use the spend column instead."))
    elif abs(cpc - 0.575452) > 0.005:
        results.append((False, f"overall_cpc should be the year's spend over its clicks, about $0.58. It is {cpc!r}."))
    else:
        results.append((True, f"2025's CTR is {ctr:.2%} and its CPC ${cpc:.2f}, from the year's totals."))

    top = found.get("top_cpc")
    if not isinstance(top, pd.DataFrame):
        results.append((False, "top_cpc isn't a DataFrame yet. ads_2025.nlargest(5, \"cpc\") gives the five rows with the highest CPC."))
    elif len(top) != 5:
        results.append((False, f"top_cpc has {len(top):,} rows, but it should have five."))
    elif list(top.columns) != ["date", "campaign_name", "clicks", "spend", "cpc"]:
        results.append((False, f"top_cpc's columns should be date, campaign_name, clicks, spend and cpc, in that order. They are {list(top.columns)}."))
    elif abs(top["cpc"].iloc[0] - 1.3204) > 0.0001:
        results.append((False, f"top_cpc's first CPC is {top['cpc'].iloc[0]:.4f}, but the highest is 1.3204. Sort by cpc, highest first."))
    elif not top["cpc"].is_monotonic_decreasing:
        results.append((False, "top_cpc has the right rows, but not highest first."))
    else:
        results.append((True, "top_cpc's most expensive day: February 8, 2025, 2025_ca_search_brand, 25 clicks at $1.32."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
ads_2025 = ads[ads["date"].dt.strftime("%Y") == "2025"].copy()
ads_2025["spend"] = ads_2025["cost_micros"] / 1_000_000
ads_2025["ctr"] = ads_2025["clicks"] / ads_2025["impressions"]
ads_2025["cpc"] = ads_2025["spend"] / ads_2025["clicks"]
ads_2025.head()
```

```python
total_spend = round(ads_2025["spend"].sum(), 2)
overall_ctr = ads_2025["clicks"].sum() / ads_2025["impressions"].sum()
overall_cpc = ads_2025["spend"].sum() / ads_2025["clicks"].sum()
f"Spend ${total_spend:,.2f}, CTR {overall_ctr:.2%}, CPC ${overall_cpc:.2f}"
```

```python
top_cpc = ads_2025.nlargest(5, "cpc")[["date", "campaign_name", "clicks", "spend", "cpc"]]
top_cpc
```

</details>